# Simple PID loop double check
Craig Lage - 20Mar26

In [ ]:
import numpy as np
import pickle as pkl
import pandas as pd
import matplotlib.pyplot as plt
from lsst.ts.ofc.state_estimator import StateEstimator
from lsst.ts.ofc.ofc import OFC
from lsst.ts.ofc import OFCData


## Get the Nightly report data and the PID logs.

In [ ]:
day_obs = 20260325

# Load the nightly table.
# The parquet file was created with: 
# https://github.com/lsst-sitcom/ts_aos_analysis/blob/tickets/DM-54406/notebooks/nightly_report/nightly_report_ts_version.ipynb
parquet_file = f"/home/c/cslage/u/MTAOS/times_square_reports/nightly_aos_table_{day_obs}.parquet"
table = pd.read_parquet(parquet_file)
print(f'Loaded {parquet_file}: {len(table)} rows')
print(f'Columns: {sorted(table.columns.tolist())}')

# Load the PID logs
# This was created with PID_Corrections_19Mar26.ipynb
filename = f"/home/c/cslage/u/MTAOS/times_square_reports/PID_data_{day_obs}.pkl"
with open(filename, 'rb') as f:
    pidDict = pkl.load(f)
    

## Create the controller

In [ ]:
ofcData = OFCData()
ofcData.configure_controller()
await ofcData.configure_instrument("lsst")
ofc = OFC(ofcData)


In [ ]:
all_labels = ['M2 dz', 'M2 dx', 'M2 dy', 'M2 rx', 'M2 ry',
     'Cam dz', 'Cam dx', 'Cam dy', 'Cam rx', 'Cam ry',
     'M1M3-1', 'M1M3-2', 'M1M3-3', 'M1M3-4', 'M1M3-5',
     'M1M3-6', 'M1M3-7', 'M1M3-8', 'M1M3-9', 'M1M3-10',
     'M1M3-11', 'M1M3-12', 'M1M3-13', 'M1M3-14', 'M1M3-15',
     'M1M3-16', 'M1M3-17', 'M1M3-18', 'M1M3-19', 'M1M3-20',
     'M2-1', 'M2-2', 'M2-3', 'M2-4', 'M2-5',
     'M2-6', 'M2-7', 'M2-8', 'M2-9', 'M2-10',
     'M2-11', 'M2-12', 'M2-13', 'M2-14', 'M2-15',
     'M2-16', 'M2-17', 'M2-18', 'M2-19', 'M2-20'
     ]

vmode_labels = ['Vmode1\nM2 tilts -rx-ry', 'Vmode2\nM2 tilts -rx+ry', 
                'Vmode3\nCam tilts -rx+ry', 'Vmode4\nCam tilts rx+ry',
                'Vmode5\nZ4-Focus', 'Vmode6\nZ5-Astig-Oblique',
                'Vmode7\nZ6-Astig-Vert', 'Vmode8\nZ7-Coma-Vert',
                'Vmode9\nZ8-Coma-Horiz', 'Vmode10\nZ9-Trefoil-Vert',
                'Vmode11\nZ10-Trefoil-Oblique', 'Vmode12\nZ11-Spherical']


## The two cells below are a verification of how the corrections are calculated.
## Basically, correction = - Kp * dof_state + Ki * PID_Integral

In [ ]:
# This is an example dof_state from 2026032500450
DOF_state = np.array([-7.69525284e+01, -6.10639338e-02,  2.36265858e-01, -9.13041173e-04,
       -3.64083977e-04, -3.08249989e+01,  7.34166702e-03, -2.81803659e-02,
       -1.04056016e-03, -1.91889829e-03,  2.92799535e-01, -3.08560693e-01,
        3.75287499e-01,  8.33104581e-03,  1.29503025e-02, -1.14993641e-01,
        3.93797551e-01,  0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
        0.00000000e+00,  0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
        0.00000000e+00,  0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
        0.00000000e+00,  0.00000000e+00,  4.43675402e-01, -3.84422576e-01,
        1.76382587e-02,  3.38663001e-02,  2.41486633e-01,  0.00000000e+00,
        0.00000000e+00,  0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
        0.00000000e+00,  0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
        0.00000000e+00,  0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
        0.00000000e+00,  0.00000000e+00])

In [ ]:
# This is a key verification.
sensor_names = ['R00_SW0', 'R00_SW1', 'R04_SW0', 'R04_SW1', 'R40_SW0', 'R40_SW1', 'R44_SW0', 'R44_SW1']
filter_name = 'R'

step = ofc.controller.control_step(filter_name, DOF_state, sensor_names, control_vmodes=False)
print(len(step))
print(step)
calc_step = -0.18 * DOF_state + 0.022 * ofc.controller.integral
print(len(calc_step))
print(calc_step)
step == calc_step

## Next, we verify that the corrections logged are what is actually being applied
## Basically dof_state(n+1) = - correction(n) + dof_state(n)

In [ ]:
# Finally, this checks - Key progress!!
indices = list(range(0, 17)) + list(range(30,35))
expId1 = 2026032500450

correction = pidDict[expId1]['Correction']
print(f"Correction {expId1}")
print(correction)
print()
expId2 = expId1 + 1
seqNum1 = int(expId1 - 1E5 * day_obs)
seq_table1 = table[table['seq'] == seqNum1]
dof_state1 = seq_table1['dof_state'].values[0]
seqNum2 = int(expId2 - 1E5 * day_obs)
seq_table2 = table[table['seq'] == seqNum2]
dof_state2 = seq_table2['dof_state'].values[0]

print(f"dof_state_{seqNum2}")
print(dof_state2[indices])
print()
print(f"dof_state_{seqNum1} - Correction")
print(dof_state1[indices] - correction)
print("These are identical within a small error")

## Next I try to calculate the correction.  These do not agree and 
## I have been unable to identify what I am doing wrong or how the correction is calculated.

In [ ]:
seqNum1 = 450
expId1 = int(1E5 * day_obs + seqNum)
indices = list(range(0, 17)) + list(range(30,35))
seq_table = table[table['seq'] == seqNum1]
dof_state = seq_table['dof_state'].values[0][indices]
print(f"dof_state seqNum{ seqNum}")
print()
corr = pidDict[expId1 + 3]['Correction']
print(f"Correction {expId1 + 3}")
print(corr)
print()
clip_i = pidDict[2026032500450]['Clipped_I']
calculated_correction = - dof_state * 0.18 + clip_i * 0.022
print(f"Calculated correction {expId1 + 3}")
print(calculated_correction)